In [ ]:
import requests
import time
import pandas as pd
from google.colab import userdata

# 🔑 Credentials
endpoint = "https://doc-intelligence-capstone.cognitiveservices.azure.com/"
key = userdata.get('DOCUMENT_INTELLIGENCE_KEY')

analyze_url = f"{endpoint}/formrecognizer/documentModels/prebuilt-document:analyze?api-version=2023-07-31"

headers = {
    "Content-Type": "application/json",
    "Ocp-Apim-Subscription-Key": key
}

# 📄 YOUR 3 DOCUMENT URLs
document_urls = [
    "https://github.com/codewithNinjaPrince/Capstone-Project-Document-Intelligence/blob/main/claim_form-1.png?raw=true",
    "https://github.com/codewithNinjaPrince/Capstone-Project-Document-Intelligence/blob/main/claim_form-2.png?raw=true",
    "https://github.com/codewithNinjaPrince/Capstone-Project-Document-Intelligence/blob/main/claim_form-3.png?raw=true"
]

# 📦 Store all extracted records
all_records = []

# 🔍 Helper function
def find_field(extracted_data, keyword):
    for k, v in extracted_data.items():
        if keyword.lower() in k.lower():
            return v
    return None

# 🔁 LOOP THROUGH ALL DOCUMENTS
for idx, document_url in enumerate(document_urls):

    print(f"\n🚀 Processing Document {idx+1}...\n")

    data = {"urlSource": document_url}

    # STEP 1: Send request
    response = requests.post(analyze_url, headers=headers, json=data)

    if response.status_code != 202:
        print("❌ Error:", response.text)
        continue

    operation_url = response.headers["operation-location"]

    # STEP 2: Poll
    while True:
        result = requests.get(operation_url, headers={
            "Ocp-Apim-Subscription-Key": key
        })

        result_json = result.json()
        status = result_json["status"]

        if status == "succeeded":
            break
        elif status == "failed":
            print("❌ Failed")
            break

        time.sleep(2)

    # STEP 3: Extract key-value pairs
    kv_pairs = result_json["analyzeResult"].get("keyValuePairs", [])
    extracted_data = {}

    for kv in kv_pairs:
        key_text = kv.get("key", {}).get("content", "").strip()
        value_text = kv.get("value", {}).get("content", "").strip()

        if key_text:
            extracted_data[key_text] = value_text

    # STEP 4: Extract required fields
    name = find_field(extracted_data, "name")
    amount = find_field(extracted_data, "amount")
    date = find_field(extracted_data, "date")
    policy = find_field(extracted_data, "policy")
    location = find_field(extracted_data, "location")
    previous_claims = find_field(extracted_data, "previous")
    claim_type = find_field(extracted_data, "type")

    # Clean amount
    if amount:
        amount_clean = amount.replace("₹", "").replace(",", "").strip()
        try:
            amount = int(amount_clean)
        except:
            pass

    # Clean previous claims
    try:
        previous_claims = int(previous_claims)
    except:
        previous_claims = 0

    # 🎯 FINAL STRUCTURED RECORD
    record = {
        "Name": name,
        "Claim_Amount": amount,
        "Date": date,
        "Policy_Number": policy,
        "Location": location,
        "Previous_Claims": previous_claims,
        "Claim_Type": claim_type,

        # ⚠️ REQUIRED FOR ML (ADD MANUALLY)
        "Fraud": "No"   # Change based on logic/data
    }

    print("✅ Extracted:", record)

    all_records.append(record)

# 📊 CONVERT TO DATAFRAME
df = pd.DataFrame(all_records)

# 💾 SAVE FILES
df.to_csv("claims_dataset.csv", index=False)

import json
with open("claims_dataset.json", "w") as f:
    json.dump(all_records, f, indent=4)

print("\n🎉 ALL DOCUMENTS PROCESSED!")
print("\n📁 Files Generated:")
print("✔ claims_dataset.csv")
print("✔ claims_dataset.json")

print("\n📊 Preview:")
print(df.head())


🚀 Processing Document 1...

✅ Extracted: {'Name': 'Rahul Sharma', 'Claim_Amount': 75000, 'Date': '15/01/2025', 'Policy_Number': 'POL123456', 'Location': 'Delhi', 'Previous_Claims': 2, 'Claim_Type': 'Medical', 'Fraud': 'No'}

🚀 Processing Document 2...

✅ Extracted: {'Name': 'Amit Kumar', 'Claim_Amount': 64000, 'Date': '17/01/2025', 'Policy_Number': 'POL567890', 'Location': 'Bangalore', 'Previous_Claims': 1, 'Claim_Type': 'Theft', 'Fraud': 'No'}

🚀 Processing Document 3...

✅ Extracted: {'Name': 'Priya Mehta', 'Claim_Amount': 75000, 'Date': '16/01/2025', 'Policy_Number': 'POL789012', 'Location': 'Mumbai', 'Previous_Claims': 2, 'Claim_Type': 'Accident', 'Fraud': 'No'}

🎉 ALL DOCUMENTS PROCESSED!

📁 Files Generated:
✔ claims_dataset.csv
✔ claims_dataset.json

📊 Preview:
           Name  Claim_Amount        Date Policy_Number   Location  \
0  Rahul Sharma         75000  15/01/2025     POL123456      Delhi   
1    Amit Kumar         64000  17/01/2025     POL567890  Bangalore   
2   Priya M

In [ ]:
import requests
import time
from google.colab import userdata

# 🔑 Credentials
endpoint = "https://doc-intelligence-capstone.cognitiveservices.azure.com/"
key = userdata.get('DOCUMENT_INTELLIGENCE_KEY')

# 📄 Document URL (GitHub RAW)
document_url = "https://raw.githubusercontent.com/codewithNinjaPrince/Capstone-Project-Document-Intelligence/main/Screenshot%202026-04-18%20102610.png"

# 📡 API URL
analyze_url = f"{endpoint}/formrecognizer/documentModels/prebuilt-document:analyze?api-version=2023-07-31"

headers = {
    "Content-Type": "application/json",
    "Ocp-Apim-Subscription-Key": key
}

data = {
    "urlSource": document_url
}

# 🚀 STEP 1: Send request
response = requests.post(analyze_url, headers=headers, json=data)

print("Status Code:", response.status_code)

if response.status_code != 202:
    print("Error:", response.text)
    exit()

# 📍 Get polling URL
operation_url = response.headers["operation-location"]
print("Polling URL:", operation_url)

# 🔁 STEP 2: Poll until result is ready
while True:
    result = requests.get(operation_url, headers={
        "Ocp-Apim-Subscription-Key": key
    })

    result_json = result.json()
    status = result_json["status"]

    print("Current Status:", status)

    if status == "succeeded":
        print("\n✅ Analysis Complete!\n")
        break
    elif status == "failed":
        print("❌ Analysis Failed")
        exit()

    time.sleep(2)

# 📊 STEP 3: Extract Key-Value Pairs
print("\n📌 Extracted Key-Value Pairs:\n")

kv_pairs = result_json["analyzeResult"].get("keyValuePairs", [])

extracted_data = {}

for kv in kv_pairs:
    key_text = kv.get("key", {}).get("content", "").strip()
    value_text = kv.get("value", {}).get("content", "").strip()

    if key_text:
        print(f"{key_text} : {value_text}")
        extracted_data[key_text] = value_text

# 🎯 STEP 4: Find Important Fields (Dynamic Search)
print("\n🎯 Important Extracted Fields:\n")

def find_field(keyword):
    for k, v in extracted_data.items():
        if keyword.lower() in k.lower():
            return v
    return "Not Found"

name = find_field("name")
amount = find_field("amount")
date = find_field("date")
policy = find_field("policy")
location = find_field("location")
previous_claims = find_field("previous")

# 💰 Clean amount if found
if amount != "Not Found":
    amount_clean = amount.replace("₹", "").replace(",", "").strip()
    try:
        amount = int(amount_clean)
    except:
        pass

# 🧾 Final Output
print("Name:", name)
print("Claim Amount:", amount)
print("Date:", date)
print("Policy Number:", policy)
print("Location:", location)
print("Previous Claims:", previous_claims)
# print(response.json())

Status Code: 202
Polling URL: https://doc-intelligence-capstone.cognitiveservices.azure.com/formrecognizer/documentModels/prebuilt-document/analyzeResults/8c367a4b-993d-4bdb-a10a-39044ba1227e?api-version=2023-07-31
Current Status: running
Current Status: succeeded

✅ Analysis Complete!


📌 Extracted Key-Value Pairs:

Policy Number: : POL123456
Customer Name: : Rahul Sharma
Date: : 12 March 2026
Claim Details: : 
Claim Amount: : ₹75,000
Claim Type: : Vehicle Damage
Location: : Delhi
Previous Claims: : 3
Description: : The vehicle was damaged due to an accident on NH-48. Front bumper and headlights need replacement.
Signature: : Rahul Sharma

🎯 Important Extracted Fields:

Name: Rahul Sharma
Claim Amount: 75000
Date: 12 March 2026
Policy Number: POL123456
Location: Delhi
Previous Claims: 3


In [ ]:
import requests
import json
from google.colab import userdata
search_key=userdata.get('PRIMARY_ADMIN_KEY')

search_service = "https://insurancesearch.search.windows.net"
index_name = "claims-index"
api_key = search_key

url = f"{search_service}/indexes/{index_name}/docs/index?api-version=2021-04-30-Preview"

headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

# 🔥 Your extracted data
document = {
    "value": [
        {
            "@search.action": "upload",
            "id": "1",
            "policy_number": "POL123456",
            "name": "Rahul Sharma",
            "date": "12 March 2026",
            "claim_amount": 75000,
            "location": "Delhi",
            "previous_claims": "3"
        }
    ]
}

response = requests.post(url, headers=headers, data=json.dumps(document))

print(response.status_code)
print(response.json())

200
{'@odata.context': "https://insurancesearch.search.windows.net/indexes('claims-index')/$metadata#Collection(Microsoft.Azure.Search.V2021_04_30_Preview.IndexResult)", 'value': [{'key': '1', 'status': True, 'errorMessage': None, 'statusCode': 200}]}


In [ ]:
search_url = "https://insurancesearch.search.windows.net/indexes/claims-index/docs/search?api-version=2021-04-30-Preview"
from google.colab import userdata
search_key=userdata.get('PRIMARY_ADMIN_KEY')

headers = {
    "Content-Type": "application/json",
    "api-key": search_key
}

query = {
    "search": "Rahul Sharma"
}

response = requests.post(search_url, headers=headers, json=query)

print(response.json())

{'@odata.context': "https://insurancesearch.search.windows.net/indexes('claims-index')/$metadata#docs(*)", 'value': [{'@search.score': 0.32716757, 'id': '1776493428', 'policy_number': 'POL123456', 'name': 'Rahul Sharma', 'date': '12 March 2026', 'claim_amount': 75000, 'location': 'Delhi', 'previous_claims': '3'}, {'@search.score': 0.32716757, 'id': '1', 'policy_number': 'POL123456', 'name': 'Rahul Sharma', 'date': '12 March 2026', 'claim_amount': 75000, 'location': 'Delhi', 'previous_claims': '3'}]}


In [ ]:
import requests
import json
from google.colab import userdata

key = userdata.get('FRAUD_KEY')

url = "http://1e9bce23-4eeb-4d0a-a792-5ed2c8c9d423.koreacentral.azurecontainer.io/score"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {key}"
}

data = {
    "Inputs": {
        "WebServiceInput0": [
            {
                "Customer_ID": "C001",
                "Claim_Amount": 70000,
                "Claim_Type": "Vehicle",
                "Location": "Delhi",
                "Previous_Claims": 3,
                "Fraud": "No"   # ✅ REQUIRED (dummy value)
            }
        ]
    },
    "GlobalParameters": {}
}

response = requests.post(url, headers=headers, json=data)

print(response.json())

{'Results': {'WebServiceOutput0': [{'Customer_ID': 'C001', 'Claim_Amount': 70000, 'Claim_Type': 'Vehicle', 'Location': 'Delhi', 'Previous_Claims': 3, 'Fraud': 'No', 'Scored Labels': 'No', 'Scored Probabilities': 0.3217698474645642}]}}
